Testing inference with guided diffusion

Constraints:

- vel should be <= 0.2
- acc should be <= 0.4

Since it's normalized to canvas size [0,1], we're going scale it up by 1.5x, so we need to scale down the vel/acc by 1.5x.  We originally had 0.3 and 0.6, which we scaled down to 0.2, 0.4

We'll use loss terms:
- relu[vel - vel_max]
- relu[acc - acc_max]

Assume trajectory is executed at 50Hz, so 0.02s per point.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import torch
import torch.nn as nn
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import PushTStateDataset, GmlDataset, normalize_data
from style.diffusion_policy_gml.network import MemorizationModel, ConditionalUnet1D, compute_noise, compute_orig
from style.diffusion_policy_gml.env import PaintingEnv
import style.diffusion_policy_gml.network as network
from style.diffusion_policy_gml.utils import plot_traj
import toppra

# General
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
import gerry

In [ ]:
pred_horizon = 512
obs_horizon = 1  # pad end but don't pad start, to discourage standing still at start
action_horizon = 1
obs_dim = 0  # x/y
action_dim = 5  # dx/dy/penup

num_diffusion_iters = 100

## Load Dataset

In [ ]:
# dataset_path = "data/gml_000000.zarr"
# dataset_path = "data/gml_003000.zarr"
dataset_path = "data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr"

# create dataset from file
# dataset = PushTStateDataset(
#     dataset_path=dataset_path,
#     pred_horizon=pred_horizon,
#     obs_horizon=obs_horizon,
#     action_horizon=action_horizon,
#     action_delta=True
# )
with gerry.Stopwatch("Loading dataset"):
    dataset = GmlDataset(
        dataset_path=dataset_path,
        sequence_length=pred_horizon,
        pad_before=0,
        pad_after=0,
        # stride=10,
        action_delta=True,
        action_penlift=True,
        normalize=dict(obs=False, action=True),
        # max_drawings=100
    )
print(f'The number of drawings is {len(dataset.episode_ends)}')
print(dataset.indices.shape)
# print(dataset.episode_ends)
# print(dataset.indices)

# create dataloader
with gerry.Stopwatch("Creating dataloader"):
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=128,
        num_workers=1,
        shuffle=True,
        pin_memory=True,
        persistent_workers=True
    )

# visualize data in batch
print("Num batches:          ", len(dataloader))
batch = next(iter(dataloader))
print("batch['obs'].shape:   ", batch['obs'].shape)
print("batch['action'].shape:", batch['action'].shape)

## Diffusion Setup

In [ ]:
# Noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    clip_sample_range=5,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

In [ ]:
# Network!
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=0,
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)
noise_pred_net.apply(init_weights);

In [ ]:
# Test with example inputs
noised_action = torch.randn((1, pred_horizon, action_dim))
obs = torch.zeros((1, obs_horizon, obs_dim))
diffusion_iter = torch.zeros((1,), dtype=torch.long)

# compute
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs.flatten(start_dim=1))

# check denoising
denoised_action = noised_action - noise

# device transfer
device = torch.device('cuda')
_ = noise_pred_net.to(device)

## Loss Function

In [ ]:
dataset.stats['action']

In [ ]:
action_stats = {k: torch.from_numpy(v).to(device) for k, v in dataset.stats['action'].items()}
for v in action_stats.values():
    v.requires_grad = False
def unnormalize_action(data):
    return normalize_data(data, action_stats, center=not dataset.action_delta) if dataset.normalize['action'] else data

def vel_and_acc(action_unnorm):
    DT = 0.02
    vel = action_unnorm[:, :, :2] / DT
    acc = torch.diff(vel, axis=1) / DT
    return vel, acc

def loss_fn(traj):
    VEL_MAX = 0.2
    ACC_MAX = 0.4
    # traj has shape [batch, pred_horizon, action_dim]
    act = unnormalize_action(traj[:, :, 2:])
    vel, acc = vel_and_acc(act)
    # ignore travel strokes
    penup = act[:, :, 2] > 0.5
    vel[penup] = 0
    acc[penup[:, :-1] | penup[:, 1:]] = 0
    loss = (torch.nn.functional.relu(torch.abs(acc) - ACC_MAX).mean() +
            torch.nn.functional.relu(torch.abs(vel) - VEL_MAX).mean())
    return loss

In [ ]:
xvels = np.linspace(-0.001, 0.001)
traj = np.stack([np.zeros_like(xvels), np.zeros_like(xvels), xvels, xvels, np.zeros_like(xvels)]).T[None, ...]
traj = torch.from_numpy(traj).to(device)
print(traj.shape)
losses = [loss_fn(traj[:, k:k+3, :]).item() for k in range(45)]
print(losses)
plt.plot(xvels[:45], losses)

## Inference

In [ ]:
if True:
    # load pretrained weights
    # This is default, training on 3000 drawings
    folder1 = f'runs/Apr04_21-01-28_eagle'
    # # training on 100 drawings
    # folder1 = f'runs/Apr05_16-14-37_eagle'
    # # training with 512-length trajectories
    # folder1 = 'runs/Apr05_16-48-06_eagle'

    device = torch.device('cuda')
    ema_noise_pred_net = ConditionalUnet1D(
        input_dim=action_dim,
        global_cond_dim=obs_dim*obs_horizon,
    )
    ema_noise_pred_net.to(device)
    ema_noise_pred_net.load_state_dict(torch.load(f'{folder1}/ema_noise_pred_net.pth'))
    global_cond = None

In [ ]:
def guidance(x):
    # Returns grad of loss function w.r.t. x
    # copy x and require grad
    x = x.clone().detach().to(device)
    x.requires_grad = True
    # compute loss
    with torch.enable_grad():
        loss = loss_fn(x) * 1e1
    # compute grad
    grad = torch.autograd.grad(loss, x)[0]
    return grad

In [ ]:
# B = 15  # num samples
B = 6*6  # num samples
all_obs = {}
all_actions = {}
all_histories = {}

# for horizon in tqdm([40, 80, 160, 320, 640, 1280, 2560]):
for horizon in tqdm([2560]):
    # action_n_init = torch.randn((B, pred_horizon, action_dim), device=device)
    action_n_init = torch.randn((B, horizon, action_dim), device=device)
    history = []

    action_n = network.eval(ema_noise_pred_net, noise_scheduler, action_n_init,
                            global_cond=global_cond[[0]] if global_cond is not None else None,
                            log_history=history,
                            guidance=guidance)
    # )

    action_n = action_n.detach().cpu().numpy()
    action = dataset.unnormalize_action(action_n[..., -3:])

    all_actions[horizon] = action
    all_histories[horizon] = history
    all_obs[horizon] = dataset.unnormalize_obs(action_n[..., :-3])

In [ ]:
scale = 1
# Plot trajectories
r, c = (B - 1) // 6 + 1, 6
fig, axes = plt.subplots(r, c, figsize=(12 / scale, 2.5 * r / scale))
fig.subplots_adjust(hspace=0.1, wspace=0.1)
acts = all_actions[40*64]
obss = all_obs[40*64]
# acts = all_actions[20]
print(acts.shape)
for act, obs, ax in zip(acts, obss, axes.flatten()):
    # ax.plot(*(obs - obs[0]).T, 'k.-')
    # plot_traj(ax, dataset.normalize_action(act), obs=obs, markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5))
    plot_traj(ax, dataset.normalize_action(act), x0=obs[0], markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5), clean=True)
    ax.axis('equal')
    # ax.set_xlim([0, 1])
    def zoom_lims(lims, factor):
        return (lims - np.mean(lims)) / factor + np.mean(lims)
    # ax.set_xlim(zoom_lims(ax.get_xlim(), 1.5))
    # ax.set_ylim(zoom_lims(ax.get_ylim(), 1.5))
fig.suptitle(f'T = {act.shape[0]}', fontsize=64 / scale);

In [ ]:
scale = 1
# Plot trajectories
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
acts = all_actions[40*64]
obss = all_obs[40*64]
# acts = all_actions[20]
print(acts.shape)
selected = [2, 3, 4, 6, 8, 9, 11, 14, 15]
ranges = [-1, -1, -1, -1, -1, -1, -1, -1, -1]
for act, obs, ax, rangee in zip(acts[selected], obss[selected], axes.flatten(), ranges):
    # ax.plot(*(obs - obs[0]).T, 'k.-')
    # plot_traj(ax, dataset.normalize_action(act), obs=obs, markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5))
    plot_traj(ax, dataset.normalize_action(act[:rangee]), x0=obs[0], markersize=.5, linewidth=0.25, travel_kwargs=dict(linewidth=0.5), line_ls='k.-')
    ax.axis('equal')
    # ax.set_xlim([0, 1])
    def zoom_lims(lims, factor):
        return (lims - np.mean(lims)) / factor + np.mean(lims)
    # ax.set_xlim(zoom_lims(ax.get_xlim(), 1.5))
    # ax.set_ylim(zoom_lims(ax.get_ylim(), 1.5))
    # ax.axis('off')
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('Approach 3: Classifier-Guided DDPM Outputs', fontsize=32)
fig.set_tight_layout(True)
# fig.savefig('results/figs/classifier_guided_drawings_black.eps')
# np.savez('results/figs/classifier_guided_drawings_black.npz', acts=acts, obss=obss, selected=selected, ranges=ranges)

In [ ]:
# Plot velocities and accelerations
if True:
    acts = all_actions[40*64]
    vels, accs = vel_and_acc(torch.from_numpy(acts))
    t = np.arange(0, act.shape[0]) * 0.02
    vel, acc = vels[0], accs[0]
else:
    with np.load('results/figs/vel_acc-classifier-guided.npz') as data:
        t = data['t']
        vels = data['vels']
        accs = data['accs']

print(t.shape, vel.shape, acc.shape)

import matplotlib as mpl
new_color_cycle = ['r', 'g', 'b', 'k']
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=new_color_cycle)

fig, axes = plt.subplots(2, 1, figsize=(6, 4), sharex=True)
axes[0].plot(t, vel, linewidth=1)
axes[1].plot(t[:-1], acc, linewidth=1)

axes[0].hlines([-.2, .2], t[0], t[-1], 'k', 'dashed')
axes[1].hlines([-.4, .4], t[0], t[-1], 'k', 'dashed')

axes[0].set_xlim(xmin=0)
axes[0].set_ylim(-.2 * 2, .2 * 2)
axes[1].set_ylim(-.4 * 2, .4 * 2)

fig.suptitle('Control Limits Adherance for Approach 3: Classifier-Guided Diffusion')
axes[1].set_xlabel('Time (s)')
axes[0].set_ylabel('Velocity (m/s)')
axes[1].set_ylabel('Acceleration (m/s)')
axes[0].legend(['x', 'y', 'limit'], loc='lower right')
fig.set_tight_layout(True)

# fig.savefig('results/figs/vel_acc-classifier-guided.eps')
# np.savez('results/figs/vel_acc-classifier-guided.npz', t=t, vels=vels, accs=accs)